# 임베딩

In [23]:
import json
import re
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer

# 3-Idiots/Project 에서 실행
PROJECT_DIR = Path(".").resolve()
REPO_ROOT = PROJECT_DIR.parent
ENV_FILE = REPO_ROOT / ".env"

MOVIE_CSV = PROJECT_DIR / "tmdb_movie_data.csv"
WEBTOON_CSV = PROJECT_DIR / "aladin_webtoon_data.csv"
OUTPUT_DIR = PROJECT_DIR / "embeddings"

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 32
FIELDS = ("title", "genres", "creator", "plot")
UPLOAD_CHUNK = 500

In [24]:
def txt(value) -> str:
    return "" if pd.isna(value) else str(value).strip()


def clean_title(title: str) -> str:
    """웹툰 상품명 → 시리즈명 (aladin_crawler 와 동일)."""
    t = txt(title)
    t = re.sub(r"[\(\[].*?[\)\]]", "", t)
    t = re.sub(r"\s+[-/:]\s+.*$", "", t)
    t = re.sub(r"\s*\d+\s*[~-]\s*\d+.*$", "", t)
    t = re.sub(
        r"\s*(\+|특전|세트|단행본|합본|박스|패키지|특별판|한정판|부록|에디션|초판|아크릴|스티커|포스터).*$",
        "",
        t,
    )
    t = re.sub(r"\s*(시즌|part)?\s*\d+\s*(권|부|화|기)?\s*$", "", t, flags=re.IGNORECASE)
    return t.strip()


def prepare_records(df: pd.DataFrame, media: str) -> list[dict]:
    records = []
    for _, row in df.iterrows():
        if media == "movie":
            director, cast = txt(row["director"]), txt(row["cast"])
            creator = f"감독: {director}, 출연: {cast}"
            keywords, overview = txt(row["keywords"]), txt(row["overview"])
            plot = f"키워드: {keywords}. {overview}" if keywords else overview
            records.append(
                {
                    "source_id": str(int(row["movie_id"])),
                    "title": txt(row["title"]),
                    "genres": txt(row["genres"]),
                    "creator": creator,
                    "plot": plot,
                }
            )
        else:
            item_id = re.search(r"ItemId=(\d+)", txt(row["link"])).group(1)
            author = txt(row["author"])
            records.append(
                {
                    "source_id": item_id,
                    "title": clean_title(row["series_title"]),
                    "genres": txt(row["genres"]),
                    "creator": f"작가: {author}",
                    "plot": txt(row["description"]),
                }
            )
    return records


def full_text(record: dict) -> str:
    return f"제목: {record['title']} | 장르: {record['genres']} | {record['creator']} | {record['plot']}"


def encode_pairs(pairs: list[tuple[str, str]], model) -> tuple[list[str], list[str], list[list[float]], int]:
    source_ids, texts = zip(*pairs)
    vectors = model.encode(
        list(texts),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    dim = len(vectors[0])
    return list(source_ids), list(texts), [v.tolist() for v in vectors], dim


def build_column_items(
    media: str, field: str, records: list[dict], model, model_name: str
) -> list[dict]:
    pairs = [(r["source_id"], r[field]) for r in records if r[field]]
    if not pairs:
        return []
    source_ids, texts, embeddings, dim = encode_pairs(pairs, model)
    return [
        {
            "media_type": media,
            "source_id": sid,
            "field": field,
            "text": text,
            "embedding": emb,
            "model": model_name,
            "dim": dim,
        }
        for sid, text, emb in zip(source_ids, texts, embeddings)
    ]


def build_full_items(media: str, records: list[dict], model, model_name: str) -> list[dict]:
    pairs = [(r["source_id"], full_text(r)) for r in records]
    source_ids, texts, embeddings, dim = encode_pairs(pairs, model)
    return [
        {
            "media_type": media,
            "source_id": sid,
            "text": text,
            "embedding": emb,
            "model": model_name,
            "dim": dim,
        }
        for sid, text, emb in zip(source_ids, texts, embeddings)
    ]

In [25]:
movies_df = pd.read_csv(MOVIE_CSV)
webtoons_df = pd.read_csv(WEBTOON_CSV)

movie_rows = prepare_records(movies_df, "movie")
webtoon_rows = prepare_records(webtoons_df, "webtoon")

print(f"영화 {len(movie_rows)}건, 웹툰 {len(webtoon_rows)}건")
print("영화 샘플:", movie_rows[0])
print("웹툰 샘플:", webtoon_rows[0])

영화 1609건, 웹툰 309건
영화 샘플: {'source_id': '1304313', 'title': '리 크로닌의 미이라', 'genres': '스릴러/범죄, 공포/호러', 'creator': '감독: 리 크로닌, 출연: 잭 레이너, 라이아 코스타, 메이 칼라마위', 'plot': '키워드: journalist, egypt, monster, ritual, kidnapping. 한 기자의 어린 딸이 흔적도 없이 사막으로 사라진다. 그리고 8년 후, 산산이 부서진 가족은 그녀가 다시 돌아왔다는 소식에 충격을 받는다. 하지만 기쁨으로 가득해야 할 재회는 곧 살아 있는 악몽으로 변해간다.'}
웹툰 샘플: {'source_id': '393727271', 'title': '전지적 독자 시점', 'genres': '판타지', 'creator': '작가: 슬리피-C (지은이), 싱숑 (원작), UMI (각색)', 'plot': '낙원의 멸망을 딛고 비상한 키메라 드래곤. 그리고 일행 앞에 주어진 새로운 시련. 김독자는 가장 사랑하는 존재에게 죽게 될 것이다. 예언대로 됐을 뿐이야. 독자야, 나는 세상 누구보다 너를 사랑한단다. 어쩌면 나 자신보다 더. 지금부터 모든 걸 ‘다시 읽을’ 거란다.'}


## API 업로드

In [26]:
import os

import requests
from dotenv import load_dotenv

load_dotenv(ENV_FILE)
BASE_URL = os.environ["BASE_URL"].rstrip("/")
ADMIN_API_KEY = os.environ["ADMIN_API_KEY"]
API_HEADERS = {"X-API-Key": ADMIN_API_KEY}


def post_items(endpoint: str, items: list[dict]) -> int:
    if not items:
        return 0
    url = f"{BASE_URL}/admin/embeddings/{endpoint}"
    total = 0
    for i in range(0, len(items), UPLOAD_CHUNK):
        resp = requests.post(
            url,
            json={"items": items[i : i + UPLOAD_CHUNK]},
            headers=API_HEADERS,
            timeout=180,
        )
        if not resp.ok:
            raise RuntimeError(f"{resp.status_code} {endpoint} {resp.text}")
        total += resp.json()["upserted"]
    return total


def upload_embeddings(model, model_name: str) -> None:
    for media, records in [("movie", movie_rows), ("webtoon", webtoon_rows)]:
        items = build_full_items(media, records, model, model_name)
        n = post_items("all", items)
        print(f"[all] {media}: {n}")

    for field in FIELDS:
        for media, records in [("movie", movie_rows), ("webtoon", webtoon_rows)]:
            items = build_column_items(media, field, records, model, model_name)
            n = post_items("column", items)
            print(f"[column] {media} {field}: {n}")

모델 로딩: paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 51/51 [00:19<00:00,  2.60it/s]


[done] movie_embedding_full.csv: 1609 rows


Batches: 100%|██████████| 10/10 [00:03<00:00,  2.76it/s]

[done] webtoon_embedding_full.csv: 309 rows


In [28]:
print(f"모델 로딩: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
upload_embeddings(model, MODEL_NAME)

Batches: 100%|██████████| 51/51 [00:02<00:00, 24.98it/s]


[done] movie_embedding_title.csv: 1609 rows


Batches: 100%|██████████| 10/10 [00:00<00:00, 28.28it/s]


[done] webtoon_embedding_title.csv: 309 rows


Batches: 100%|██████████| 51/51 [00:01<00:00, 26.32it/s]


...

[done] webtoon_embedding_creator.csv: 309 rows


Batches: 100%|██████████| 51/51 [00:20<00:00,  2.53it/s]


[done] movie_embedding_plot.csv: 1609 rows


Batches: 100%|██████████| 10/10 [00:03<00:00,  3.27it/s]

[done] webtoon_embedding_plot.csv: 297 rows

필드별 저장 완료 → C:\Users\USER\Desktop\code\3-Idiots\Project\embeddings


## CSV 출력

In [29]:
def save_items_csv(path: Path, items: list[dict], field: str | None = None) -> None:
    rows = []
    for item in items:
        row = {
            "source_id": item["source_id"],
            "text": item["text"],
            "embedding": json.dumps(item["embedding"]),
            "model": item["model"],
            "dim": item["dim"],
        }
        if field:
            row["field"] = field
        rows.append(row)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[csv] {path.name}: {len(rows)} rows")


def export_all_csv(model, model_name: str) -> None:
    for media, records in [("movie", movie_rows), ("webtoon", webtoon_rows)]:
        save_items_csv(
            OUTPUT_DIR / f"{media}_embedding_full.csv",
            build_full_items(media, records, model, model_name),
        )
    for field in FIELDS:
        for media, records in [("movie", movie_rows), ("webtoon", webtoon_rows)]:
            save_items_csv(
                OUTPUT_DIR / f"{media}_embedding_{field}.csv",
                build_column_items(media, field, records, model, model_name),
                field=field,
            )


def upload_from_csv() -> None:
    for path in sorted(OUTPUT_DIR.glob("*_embedding_full.csv")):
        media = path.stem.replace("_embedding_full", "")
        df = pd.read_csv(path)
        items = [
            {
                "media_type": media,
                "source_id": str(row["source_id"]),
                "text": row["text"],
                "embedding": json.loads(row["embedding"]),
                "model": row["model"],
                "dim": int(row["dim"]),
            }
            for _, row in df.iterrows()
        ]
        print(f"[all] {path.name}: {post_items('all', items)}")

    for path in sorted(OUTPUT_DIR.glob("*_embedding_*.csv")):
        if path.name.endswith("_full.csv"):
            continue
        media, field = path.stem.split("_embedding_", 1)
        df = pd.read_csv(path)
        items = [
            {
                "media_type": media,
                "source_id": str(row["source_id"]),
                "field": field,
                "text": row["text"],
                "embedding": json.loads(row["embedding"]),
                "model": row["model"],
                "dim": int(row["dim"]),
            }
            for _, row in df.iterrows()
        ]
        print(f"[column] {path.name}: {post_items('column', items)}")

In [ ]:
# export_all_csv(model, MODEL_NAME)   # CSV 저장만
upload_from_csv()                   # 기존 CSV → API

[all] movie_embedding_full.csv: 1609
[all] webtoon_embedding_full.csv: 309
[column] movie_embedding_creator.csv: 1609
[column] movie_embedding_genres.csv: 1609
[column] movie_embedding_plot.csv: 1609
[column] movie_embedding_title.csv: 1609
[column] webtoon_embedding_creator.csv: 309
[column] webtoon_embedding_genres.csv: 309
[column] webtoon_embedding_plot.csv: 297
[column] webtoon_embedding_title.csv: 309
